# Phần 4 – Câu 2: Quốc doanh và tư nhân có khác biệt về NIM, NPL, CAR?
Nhóm 5 PTDL · Trực quan hóa & Phân tích thống kê

**Dữ liệu:** bảng 4 – `nim`, `npl`, `car`, `car_nguon`, `loai_hinh_so_huu`, `ma_nh`, `nam`.
**Nhóm:** quốc doanh = VCB, BID, CTG; tư nhân = TCB, MBB, ACB, STB, VPB.

| Bước | Việc trong kế hoạch |
|---|---|
| 1 | Thống kê mô tả NIM, NPL, CAR theo hai nhóm |
| 2 | Chuẩn bị dữ liệu kiểm định: NIM, NPL → 40 quan sát ngân hàng–năm; CAR → 32 giá trị công bố |
| 3 | Kiểm tra điều kiện: Shapiro-Wilk (phân phối chuẩn), Levene (phương sai) |
| 4 | Chọn kiểm định và trình bày H0/H1, thống kê, p-value, kết luận ở mức 5% |
| 5 | Kiểm tra lại: NPL không có VPB (thêm: CAR chỉ dùng số VNDirect, kiểm định ở cấp ngân hàng) |
| 6 | Biểu đồ: boxplot hai nhóm, line trung bình hai nhóm theo năm |
| 7 | Xuất `kiem_dinh_nhom.csv`, trả lời câu 2 kèm hạn chế |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from scipy import stats
from src import config as cf
from src.data_loader import doc_tong_hop
from src.thong_ke import bang_mo_ta, du_lieu_cau2, kiem_dinh_nhom
from src.ket_qua import xuat_ket_qua
from src.viz import ve_hop_nhom, ve_duong_nhom, ve_cot_trung_binh_nam

b4 = doc_tong_hop()
ALPHA = 0.05

LUU_HINH = True
def hien(fig, ten=None, show=True):
    global LUU_HINH
    if show:
        fig.show()
    if LUU_HINH and ten:
        try:
            cf.luu_hinh(fig, ten)
        except Exception as e:
            LUU_HINH = False
            print("Không lưu được PNG (tạm tắt lưu hình):", str(e).splitlines()[0])

---
## 1. Thống kê mô tả theo hai nhóm
NIM, NPL dùng 160 quan sát quý; CAR dùng 32 giá trị công bố.

In [2]:
mo_ta_p4 = pd.concat([
    bang_mo_ta(b4, ["nim", "npl"], cac_cap_do=(None, "loai_hinh_so_huu", "ma_nh", "nam")),
    bang_mo_ta(b4[b4["car_cong_bo"]], ["car"], cac_cap_do=(None, "loai_hinh_so_huu", "ma_nh", "nam"),
               mau_du_lieu="car_cong_bo"),
], ignore_index=True)

(mo_ta_p4[mo_ta_p4["cap_do"] == "loai_hinh_so_huu"]
 .set_index(["chi_so", "loai_hinh_so_huu"])
 [["so_quan_sat", "trung_binh", "trung_vi", "do_lech_chuan", "nho_nhat", "lon_nhat"]].round(2))

so_quan_sat  trung_binh  trung_vi  do_lech_chuan  \
chi_so loai_hinh_so_huu                                                     
nim    Quốc doanh                 60        2.85      2.87           0.34   
       Tư nhân                   100        4.64      4.29           1.36   
npl    Quốc doanh                 60        1.24      1.24           0.33   
       Tư nhân                   100        2.06      1.45           1.56   
car    Quốc doanh                 12        9.65      9.24           1.05   
       Tư nhân                    20       12.64     12.15           2.40   

                         nho_nhat  lon_nhat  
chi_so loai_hinh_so_huu                      
nim    Quốc doanh            2.02      3.51  
       Tư nhân               2.02      9.09  
npl    Quốc doanh            0.58      1.98  
       Tư nhân               0.36      6.52  
car    Quốc doanh            8.90     12.16  
       Tư nhân               9.11     17.10

In [3]:
(cf.THU_MUC_KET_QUA / "tam").mkdir(parents=True, exist_ok=True)
p = xuat_ket_qua(mo_ta_p4, "thong_ke_mo_ta", thu_muc=cf.THU_MUC_KET_QUA / "tam")
p.replace(cf.THU_MUC_KET_QUA / "tam" / "thong_ke_mo_ta_p4.csv")
print("Đã lưu outputs/tam/thong_ke_mo_ta_p4.csv – Phần 6 sẽ gộp với Phần 3")

Đã lưu outputs/tam/thong_ke_mo_ta_p4.csv – Phần 6 sẽ gộp với Phần 3


**Nhìn sơ bộ:** NIM và CAR của tư nhân cao hơn rõ; NPL trung bình tư nhân cũng cao hơn (2,06% so với 1,24%) nhưng **trung vị gần bằng nhau** – dấu hiệu có vài giá trị rất cao kéo trung bình lên. Độ lệch chuẩn NPL của tư nhân lớn gấp nhiều lần quốc doanh.

---
## 2. Chuẩn bị dữ liệu kiểm định
**Vì sao gom về ngân hàng–năm?** 4 quý trong cùng một năm của một ngân hàng gần như giống nhau; dùng thẳng 160 quý sẽ làm kiểm định "tưởng" có nhiều thông tin hơn thực tế. Gom về trung bình năm giảm bớt sự lặp lại này.
**Vì sao CAR chỉ 32 giá trị?** Các quý khác là số điền tiếp, không phải số thật. 32 giá trị gồm 26 số VNDirect công bố và 6 số tra thủ công (BID, CTG 2022–2024).

In [4]:
dl = du_lieu_cau2(b4)
for ten, df in dl.items():
    print(f"{ten:7s}: {len(df):3d} quan sát –", df.groupby("loai_hinh_so_huu").size().to_dict())

nh_nam :  40 quan sát – {'Quốc doanh': 15, 'Tư nhân': 25}
car    :  32 quan sát – {'Quốc doanh': 12, 'Tư nhân': 20}
nh     :   8 quan sát – {'Quốc doanh': 3, 'Tư nhân': 5}


In [5]:
dl["nh"].round(2)   # trung bình cả giai đoạn theo ngân hàng (dùng ở bước kiểm tra lại)

,loai_hinh_so_huu,ma_nh,nim,npl,car
0,Quốc doanh,BID,2.60,1.50,9.02
1,Quốc doanh,CTG,2.88,1.30,9.24
2,Quốc doanh,VCB,3.07,0.92,10.70
3,Tư nhân,ACB,3.84,1.08,12.08
4,Tư nhân,MBB,4.87,1.47,11.02
5,Tư nhân,STB,3.35,2.09,9.67
6,Tư nhân,TCB,4.64,0.92,14.98
7,Tư nhân,VPB,6.53,4.72,15.45


---
## 3. Kiểm tra điều kiện và chọn kiểm định

**Quy tắc chọn** (theo kế hoạch):
```
Shapiro-Wilk cả 2 nhóm p ≥ 0,05 (phân phối chuẩn)?
├── Có → Levene p ≥ 0,05 (phương sai bằng nhau)?
│        ├── Có    → Student t-test
│        └── Không → Welch t-test
└── Không → Mann-Whitney U (không cần phân phối chuẩn)
```
- **Shapiro-Wilk:** H0 = dữ liệu có phân phối chuẩn. p < 0,05 → không chuẩn.
- **Levene:** H0 = hai nhóm có phương sai bằng nhau. p < 0,05 → phương sai khác nhau.

In [6]:
chinh = pd.DataFrame([
    kiem_dinh_nhom(dl["nh_nam"], "nim", "chinh", "ngan_hang_nam", alpha=ALPHA),
    kiem_dinh_nhom(dl["nh_nam"], "npl", "chinh", "ngan_hang_nam", alpha=ALPHA),
    kiem_dinh_nhom(dl["car"], "car", "chinh", "ngan_hang_nam_cong_bo", alpha=ALPHA),
])
chinh[["chi_so", "n_quoc_doanh", "n_tu_nhan", "shapiro_p_quoc_doanh", "shapiro_p_tu_nhan",
       "levene_p", "kiem_dinh"]].round(4)

,chi_so,n_quoc_doanh,n_tu_nhan,shapiro_p_quoc_doanh,shapiro_p_tu_nhan,levene_p,kiem_dinh
0,nim,15,25,0.5515,0.0846,0.0036,Welch t
1,npl,15,25,0.9596,0.0003,0.0133,Mann-Whitney U
2,car,12,20,0.0013,0.1972,0.0017,Mann-Whitney U


Đọc bảng:
- **NIM:** cả hai nhóm chuẩn (0,55 và 0,08 ≥ 0,05) nhưng phương sai khác nhau (Levene 0,004) → **Welch t-test**.
- **NPL:** nhóm tư nhân không chuẩn (Shapiro p < 0,001, do các giá trị cao của VPB) → **Mann-Whitney U**.
- **CAR:** nhóm quốc doanh không chuẩn (p = 0,001: BID, CTG tập trung quanh 9%, còn VCB tăng dần tới 12%) → **Mann-Whitney U**.

---
## 4. Kết quả kiểm định chính (mức ý nghĩa 5%)

In [7]:
for _, r in chinh.iterrows():
    print(f"■ {r['chi_so'].upper()}  –  {r['kiem_dinh']}")
    print(f"  H0: {r['gia_thuyet_H0']}")
    print(f"  H1: {r['gia_thuyet_H1']}")
    print(f"  TB quốc doanh = {r['tb_quoc_doanh']:.2f}%, TB tư nhân = {r['tb_tu_nhan']:.2f}% "
          f"(trung vị {r['trung_vi_quoc_doanh']:.2f}% vs {r['trung_vi_tu_nhan']:.2f}%)")
    print(f"  Thống kê = {r['thong_ke']:.3f}, p-value = {r['p_value']:.6f}, "
          f"{r['ten_hieu_ung']} = {r['do_lon_hieu_ung']:.2f}")
    print(f"  → {r['ket_luan']}\n")

■ NIM  –  Welch t
  H0: NIM trung bình của nhóm quốc doanh bằng nhóm tư nhân
  H1: NIM trung bình của hai nhóm khác nhau
  TB quốc doanh = 2.85%, TB tư nhân = 4.64% (trung vị 2.92% vs 4.27%)
  Thống kê = -6.501, p-value = 0.000000, Cohen d = -1.69
  → Bác bỏ H0 (p < 0.001 < 0.05): NIM hai nhóm khác biệt có ý nghĩa, nhóm tư nhân cao hơn.

■ NPL  –  Mann-Whitney U
  H0: Phân phối NPL của nhóm quốc doanh và tư nhân như nhau
  H1: Phân phối NPL của hai nhóm khác nhau
  TB quốc doanh = 1.24%, TB tư nhân = 2.06% (trung vị 1.26% vs 1.48%)
  Thống kê = 148.000, p-value = 0.275911, biserial hạng r = -0.21
  → Chưa đủ cơ sở bác bỏ H0 (p = 0.276 ≥ 0.05): chưa thấy khác biệt có ý nghĩa về NPL.

■ CAR  –  Mann-Whitney U
  H0: Phân phối CAR của nhóm quốc doanh và tư nhân như nhau
  H1: Phân phối CAR của hai nhóm khác nhau
  TB quốc doanh = 9.65%, TB tư nhân = 12.64% (trung vị 9.24% vs 12.15%)
  Thống kê = 26.000, p-value = 0.000273, biserial hạng r = -0.78
  → Bác bỏ H0 (p < 0.001 < 0.05): CAR hai n

**Độ lớn hiệu ứng** cho biết khác biệt *lớn đến đâu*, không chỉ *có hay không*:
- Cohen d: |d| ≈ 0,2 nhỏ, 0,5 vừa, ≥ 0,8 lớn. NIM có d = −1,69 → khác biệt **rất lớn**.
- Biserial hạng r (−1 đến 1): |r| ≈ 0,1 nhỏ, 0,3 vừa, ≥ 0,5 lớn. CAR có r = −0,78 → lớn; NPL r = −0,21 → nhỏ.
- Dấu âm = nhóm quốc doanh thấp hơn.

### 4.1 Vì sao chọn đúng kiểm định lại quan trọng – ví dụ NPL
Nếu bỏ qua bước kiểm tra điều kiện và dùng luôn t-test cho NPL:

In [8]:
a = dl["nh_nam"].query("loai_hinh_so_huu == 'Quốc doanh'")["npl"]
b = dl["nh_nam"].query("loai_hinh_so_huu == 'Tư nhân'")["npl"]
print(f"Welch t-test (dùng sai): p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.4f}  → sẽ kết luận 'có khác biệt'")
print(f"Mann-Whitney U (đúng):   p = {stats.mannwhitneyu(a, b).pvalue:.4f}  → chưa thấy khác biệt")

Welch t-test (dùng sai): p = 0.0153  → sẽ kết luận 'có khác biệt'
Mann-Whitney U (đúng):   p = 0.2759  → chưa thấy khác biệt


t-test so sánh **trung bình**, nên bị 5 giá trị NPL rất cao của VPB (3,9–5,9%) kéo lệch. Mann-Whitney so sánh **thứ hạng**, ít bị ảnh hưởng bởi vài giá trị cực đoan. Phần 5 dưới đây kiểm tra trực tiếp vai trò của VPB.

---
## 5. Kiểm tra lại (robustness)
1. **NPL không có VPB** (theo kế hoạch): VPB hợp nhất công ty tài chính tiêu dùng nên NPL cao về cấu trúc.
2. **CAR chỉ dùng 26 số VNDirect**, bỏ 6 số tra thủ công của BID, CTG 2022–2024 – xem kết luận có phụ thuộc vào các số nhóm tự tra không.
3. **Cấp ngân hàng** (3 quốc doanh vs 5 tư nhân, mỗi ngân hàng 1 giá trị trung bình): các năm của cùng một ngân hàng không thật sự độc lập, nên kiểm tra ở mức ngân hàng là cách thận trọng nhất. Với mẫu nhỏ như vậy dùng Mann-Whitney U chính xác; p nhỏ nhất có thể đạt là 0,036 (khi 3 quốc doanh nằm hoàn toàn dưới/trên 5 tư nhân).

In [9]:
nn = dl["nh_nam"]
kiem_tra = pd.DataFrame([
    kiem_dinh_nhom(nn[nn["ma_nh"] != "VPB"], "npl", "khong_VPB", "ngan_hang_nam", "kiem_tra_lai", ALPHA),
    kiem_dinh_nhom(dl["car"][dl["car"]["nam"] <= 2024], "car", "car_2021_2024",
                   "ngan_hang_nam_cong_bo", "kiem_tra_lai", ALPHA),
    *[kiem_dinh_nhom(dl["nh"], cs, "cap_ngan_hang", "ngan_hang", "kiem_tra_lai", ALPHA, bat_buoc="Mann-Whitney U")
      for cs in ["nim", "npl", "car"]],
])
tong = pd.concat([chinh, kiem_tra], ignore_index=True)
tong[["chi_so", "phan_tich", "mau_du_lieu", "n_quoc_doanh", "n_tu_nhan", "tb_quoc_doanh", "tb_tu_nhan",
      "kiem_dinh", "p_value", "bac_bo_H0"]].round(4)

,chi_so,phan_tich,mau_du_lieu,n_quoc_doanh,n_tu_nhan,tb_quoc_doanh,tb_tu_nhan,kiem_dinh,p_value,bac_bo_H0
0,nim,chinh,chinh,15,25,2.8486,4.6446,Welch t,0.0000,True
1,npl,chinh,chinh,15,25,1.2407,2.0569,Mann-Whitney U,0.2759,False
2,car,chinh,chinh,12,20,9.6542,12.6400,Mann-Whitney U,0.0003,True
3,npl,kiem_tra_lai,khong_VPB,15,20,1.2407,1.3921,Mann-Whitney U,0.9601,False
4,car,kiem_tra_lai,car_chi_so_vndirect,6,20,10.1533,12.6400,Student t,0.0240,True
5,nim,kiem_tra_lai,cap_ngan_hang,3,5,2.8486,4.6446,Mann-Whitney U,0.0357,True
6,npl,kiem_tra_lai,cap_ngan_hang,3,5,1.2407,2.0569,Mann-Whitney U,0.5714,False
7,car,kiem_tra_lai,cap_ngan_hang,3,5,9.6542,12.6400,Mann-Whitney U,0.0714,False


Đọc bảng:
- **NIM:** khác biệt ở cả mức ngân hàng–năm và mức ngân hàng (p = 0,036 – mức tối đa có thể với 3 vs 5: NIM của cả 3 ngân hàng quốc doanh đều thấp hơn mọi ngân hàng tư nhân). → **Kết luận vững.**
- **NPL:** không khác biệt trong mọi cách kiểm tra; bỏ VPB thì TB tư nhân (1,39%) gần như bằng quốc doanh (1,24%), p = 0,96. → **Kết luận vững: không có khác biệt.**
- **CAR:** khác biệt ở mức ngân hàng–năm, kể cả khi chỉ dùng số VNDirect (p = 0,024). Nhưng ở mức ngân hàng, p = 0,071 > 0,05 vì VCB (10,7%) cao hơn STB (9,7%). → **Kết luận có, nhưng kém vững hơn NIM.**

---
## 6. Biểu đồ so sánh nhóm
### 6.1 Boxplot hai nhóm (mỗi chấm = 1 quan sát dùng trong kiểm định)

In [10]:
for _, r in chinh.iterrows():
    p_txt = "p < 0.001" if r["p_value"] < 0.001 else f"p = {r['p_value']:.3f}"
    ghi_chu = f"{r['kiem_dinh']}: {p_txt} – {'có khác biệt' if r['bac_bo_H0'] else 'chưa thấy khác biệt'} ở mức 5%"
    df = dl["car"] if r["chi_so"] == "car" else dl["nh_nam"]
    hien(ve_hop_nhom(df, r["chi_so"], ghi_chu), f"p4_hop_nhom_{r['chi_so']}")

Không lưu được PNG (tạm tắt lưu hình): 


### 6.2 Trung bình hai nhóm theo năm

In [11]:
hien(ve_duong_nhom(dl["nh_nam"], "nim"), "p4_duong_nhom_nim")
hien(ve_duong_nhom(dl["nh_nam"], "npl", bo_ma="VPB"), "p4_duong_nhom_npl")
hien(ve_duong_nhom(dl["car"], "car"), "p4_duong_nhom_car")

Đường nét đứt "Tư nhân (bỏ VPB)" ở biểu đồ NPL cho thấy: không có VPB, NPL tư nhân thấp hơn quốc doanh năm 2021–2022, rồi tăng nhanh và vượt lên từ 2023 (chủ yếu do STB, MBB).

### 6.3 Mức ngân hàng (8 chấm)

In [12]:
for cs in ["nim", "npl", "car"]:
    hien(ve_hop_nhom(dl["nh"], cs), f"p4_hop_ngan_hang_{cs}")

---
## 7. Xuất bảng kết quả

In [13]:
xuat_ket_qua(tong, "kiem_dinh_nhom")
print("Đã lưu outputs/kiem_dinh_nhom.csv:", len(tong), "dòng (3 chính + 5 kiểm tra lại)")

Đã lưu outputs/kiem_dinh_nhom.csv: 8 dòng (3 chính + 5 kiểm tra lại)


---
## 8. Trả lời câu 2 (bản nháp – viết lại bằng lời của bạn)

**Có khác biệt ở NIM và CAR, không có khác biệt ở NPL.**

- **NIM – khác biệt rõ và vững.** NIM trung bình của nhóm tư nhân 4,64%, gần gấp rưỡi quốc doanh 2,85% (Welch t-test, p < 0,001; Cohen d = −1,69, hiệu ứng rất lớn). Cả ba ngân hàng quốc doanh đều có NIM thấp hơn mọi ngân hàng tư nhân. Khoảng cách này duy trì suốt 2021–2025 dù NIM cả hai nhóm cùng giảm.
- **CAR – tư nhân cao hơn.** CAR cuối năm trung bình 12,6% (tư nhân) so với 9,7% (quốc doanh) (Mann-Whitney U, p < 0,001; r = −0,78). Kết luận giữ nguyên khi bỏ 6 số tra thủ công của BID, CTG và chỉ dùng số VNDirect (p = 0,024), nhưng không còn ý nghĩa khi so ở mức từng ngân hàng (p = 0,071) vì VCB có CAR cao hơn STB.
- **NPL – chưa thấy khác biệt.** Trung bình tư nhân cao hơn (2,06% so với 1,24%) nhưng chủ yếu do VPB; trung vị hai nhóm gần nhau. Mann-Whitney U cho p = 0,276; bỏ VPB thì p = 0,96. Nếu dùng t-test sẽ kết luận sai là có khác biệt (p = 0,015) – ví dụ cho thấy cần kiểm tra điều kiện trước khi chọn kiểm định.

**Hạn chế.**
1. Nhóm quốc doanh chỉ có 3 ngân hàng; các quan sát ngân hàng–năm của cùng một ngân hàng không hoàn toàn độc lập, nên p-value ở mức ngân hàng–năm có thể lạc quan hơn thực tế. Ở mức ngân hàng chỉ có 3 vs 5 quan sát, khả năng phát hiện khác biệt rất hạn chế.
2. CAR chỉ có 32 giá trị thật (đến 2024), trong đó 6 giá trị của BID, CTG 2022–2024 là số nhóm tra thủ công vì VNDirect ghi sai (8,00%).
3. Khác biệt nhóm phản ánh cả mô hình kinh doanh (VPB cho vay tiêu dùng, TCB vốn dày), không chỉ hình thức sở hữu – kiểm định chỉ cho biết *có* khác biệt, không giải thích *vì sao*.